# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Testing Data
testing different data structure

In [45]:
import os, torch, logging
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

from tqdm import tqdm
from flash_attn import flash_attn_func
import accelerate
from openai import OpenAI
import wandb
from huggingface_hub import login
import os

In [46]:
%load_ext dotenv
%dotenv

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)
client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"),
)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


## OpenAI API
丟整個 list 叫他改寫效果很差，一句一句來效果比較好。

In [47]:
completion = client.chat.completions.create(
  model="gpt-4-turbo",
  messages=[
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "Say , Jim , how about going for a few beers after dinner ?"}
  ],
)

print(completion.choices[0].message.content)

Jim, I'm fed up with just sitting around. How about we grab some beers after dinner and actually do something for once?


## Dataset

Observe first 10% of data from better_daily_dialog

In [48]:
idx2label = {
    0 : "neutral",
    1 : "anger",
    2 : "disgust",
    3 : "fear",
    4 : "joy",
    5 : "sadness",
    6 : "surprise"
}

In [49]:
def example(content):
    return [
        {"role": "system", "content": "rewrite with anger emotions"},
        {"role": "user", "content": content}
    ]

def format_dialogue(content, response):
    return [
        {"role": "system", "content": "rewrite with anger emotions"},
        {"role": "user", "content": content},
        {"role": "assistant", "content": response},
    ]
    
def idx_example(idx, content):
    return [
        {"role": "system", "content": "rewrite with " + idx2label[int(idx / 100)] + " emotions"},
        {"role": "user", "content": content}
    ]

def idx_format_dialogue(idx, content, response):
    return [
        {"role": "system", "content": "rewrite with " + idx2label[int(idx / 100)] + " emotions"},
        {"role": "user", "content": content},
        {"role": "assistant", "content": response},
    ]

In [50]:
# id2label = {
#     0 : "neutral",
#     1 : "anger",
#     2 : "disgust",
#     3 : "fear",
#     4 : "joy",
#     5 : "sadness",
#     6 : "surprise"
# }
# print("this is " + id2label[int(599/100)] + " emotions")

In [51]:
# chats[1]

In [52]:
# test_data = []
# for sublist in chats:
#     test_data.append(sublist[:2])

# test_data[1]

## Model
setup model

In [53]:
# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

## Data processing
paking up and shits...

Get the first 100 utterance of the dataset and send to chatGPT

In [54]:
dd = load_dataset("benjaminbeilharz/better_daily_dialog", split="train[:1%]")
dd

Dataset({
    features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
    num_rows: 872
})

took ~30 min

In [55]:
dd_list = dd[0:700]['utterance']
chats = []

pbar = tqdm(total=len(dd_list))

for i, content in enumerate(dd_list):
    completion = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=idx_example(i, content)
    )
    response = completion.choices[0].message.content
    chats.append(idx_format_dialogue(i, content, response))
    pbar.update(1)

pbar.close()


for i, chat in enumerate(chats):
    print(f"chat{i+1} = {chat}")

100%|██████████| 700/700 [28:16<00:00,  2.42s/it]

chat1 = [{'role': 'system', 'content': 'rewrite with neutral emotions'}, {'role': 'user', 'content': 'Say , Jim , how about going for a few beers after dinner ? '}, {'role': 'assistant', 'content': 'Jim, would you like to go for some beers after dinner?'}]
chat2 = [{'role': 'system', 'content': 'rewrite with neutral emotions'}, {'role': 'user', 'content': ' You know that is tempting but is really not good for our fitness . '}, {'role': 'assistant', 'content': "That option might be appealing, but it's not the best choice for our health goals."}]
chat3 = [{'role': 'system', 'content': 'rewrite with neutral emotions'}, {'role': 'user', 'content': ' What do you mean ? It will help us to relax . '}, {'role': 'assistant', 'content': 'What do you mean? It could help us relax.'}]
chat4 = [{'role': 'system', 'content': 'rewrite with neutral emotions'}, {'role': 'user', 'content': " Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? "}, {'role': 'ass

In [56]:
dataset = Dataset.from_dict({"chat": chats})
dataset = dataset.map(lambda x: {"formatted_chat": llama_tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})
print(dataset['formatted_chat'])

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

['<s>[INST] <<SYS>>\nrewrite with neutral emotions\n<</SYS>>\n\nSay , Jim , how about going for a few beers after dinner ? [/INST] Jim, would you like to go for some beers after dinner? </s>', "<s>[INST] <<SYS>>\nrewrite with neutral emotions\n<</SYS>>\n\n You know that is tempting but is really not good for our fitness . [/INST] That option might be appealing, but it's not the best choice for our health goals. </s>", '<s>[INST] <<SYS>>\nrewrite with neutral emotions\n<</SYS>>\n\n What do you mean ? It will help us to relax . [/INST] What do you mean? It could help us relax. </s>', "<s>[INST] <<SYS>>\nrewrite with neutral emotions\n<</SYS>>\n\n Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? [/INST] Do you think so? I don't agree. It might not be the healthiest choice and could affect our behavior, like it did last time. </s>", "<s>[INST] <<SYS>>\nrewrite with neutral emotions\n<</SYS>>\n\n I guess you are right.But what shall we do ? I 

In [57]:
dataset

Dataset({
    features: ['chat', 'formatted_chat'],
    num_rows: 700
})

## test data processing

In [58]:
test = load_dataset("benjaminbeilharz/better_daily_dialog", split="train[-1%:]")
test

Dataset({
    features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
    num_rows: 872
})

In [59]:
test_list = test[0:20]['utterance']
chats = []

pbar = tqdm(total=len(test_list))

for i, content in enumerate(test_list):
    # completion = client.chat.completions.create(
    #     model="gpt-4-turbo",
    #     messages=idx_example(content)
    # )
    # response = completion.choices[0].message.content
    chats.append(idx_example(100, content))
    pbar.update(1)

pbar.close()


for i, chat in enumerate(chats):
    print(f"chat{i+1} = {chat}")

100%|██████████| 20/20 [00:00<00:00, 189359.10it/s]

chat1 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': " Yes , that's true . Damn ! I just lost half my paper , and now I can't even do my homework.This is a bad time for this to happen . "}]
chat2 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': ' I have a flashlight in my closet . If you want to use that to read , you can . '}]
chat3 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': ' Thanks . I think I will try . Where are you going ? '}]
chat4 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': " I like your idea of getting a beer . I think I'll go out myself . "}]
chat5 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': " Maybe we can trade.Why don't you stay here and read for my exam , and I'll go drink beer ? "}]
chat6 = [{'role': 'system', 'content': 'rewrite with ang

In [60]:
test_dataset = Dataset.from_dict({"chat": chats})
test_dataset = test_dataset.map(lambda x: {"formatted_chat": llama_tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})
print(test_dataset['formatted_chat'])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

["<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n Yes , that's true . Damn ! I just lost half my paper , and now I can't even do my homework.This is a bad time for this to happen . [/INST]", '<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n I have a flashlight in my closet . If you want to use that to read , you can . [/INST]', '<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n Thanks . I think I will try . Where are you going ? [/INST]', "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n I like your idea of getting a beer . I think I'll go out myself . [/INST]", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n Maybe we can trade.Why don't you stay here and read for my exam , and I'll go drink beer ? [/INST]", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n No , it just won't work.If I read for your exam , you won't know the answers tomorrow . I will . [/INST]", '<s>[INST] <<SYS>>\nrewrite with anger emotions\n<<

In [61]:
test_dataset

Dataset({
    features: ['chat', 'formatted_chat'],
    num_rows: 20
})

### Saving them for later use
我不太確定能不能用 `train_test_split` 直接切一切，姑且先分開存反正很小。

In [63]:
dataset.to_json("dataset_v1.json")
test_dataset.to_json("test_dataset_v1.json")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

6880

In [64]:
# dataset.push_to_hub("Shotaro30678/candidate_generation_700")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Shotaro30678/candidate_generation_700/commit/349fe908ceb0a7af71af7e7f179efbe77f4616ef', commit_message='Upload dataset', commit_description='', oid='349fe908ceb0a7af71af7e7f179efbe77f4616ef', pr_url=None, pr_revision=None, pr_num=None)